# Figures of the manuscript

This notebook draws every figure of the manuscript with matplotlib from `results_table.xlsx` in this folder and writes them to `figures/`. The paper includes these PDFs unscaled.

* Widths: one column 216.24 pt, full text width 455.24 pt (measured in the compiled manuscript).
* All text: 9 pt, one size below the 10 pt body text, the same for labels, ticks, legends and annotations.
* `check(fig)` stops the notebook if a text has another size, leaves the figure, overlaps another text, runs into another panel, or covers a plotted point.

In [1]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.path import Path as MPath
from matplotlib.text import Text

TABLE = Path("results_table.xlsx")
OUT = Path("figures")
OUT.mkdir(exist_ok=True)

PT = 1 / 72.27            # TeX point in inches
COLUMN = 216.24 * PT      # \columnwidth
TEXT = 455.24 * PT        # \textwidth
FONT = 9                  # body text is 10 pt

plt.style.use("default")
plt.rcParams.update({k: FONT for k in ("font.size", "axes.titlesize", "axes.labelsize", "xtick.labelsize", "ytick.labelsize",
                                       "legend.fontsize", "legend.title_fontsize", "figure.titlesize", "figure.labelsize")})
plt.rcParams.update({"figure.constrained_layout.use": True, "pdf.fonttype": 42, "svg.fonttype": "none"})

sheets = pd.read_excel(TABLE, sheet_name=None)
grid, runs, lomo, gen = sheets["grid"], sheets["optimizer_runs"], sheets["lomo"], sheets["generalization"]
key = dict(zip(sheets["key_figures"].name, sheets["key_figures"].value))

MODEL = {"qwen3-vl:30b-a3b-instruct": "Qwen3-VL 30B-A3B", "gemma3n:e2b": "Gemma 3n E2B", "granite4:tiny-h": "Granite 4 Tiny-H",
         "phi4-mini:3.8b-q4_K_M": "Phi-4-mini 3.8B", "qwen3-vl:2b-instruct": "Qwen3-VL 2B", "granite3.3:2b": "Granite 3.3 2B",
         "gemma3:1b": "Gemma 3 1B", "qwen2.5-coder:0.5b-instruct": "Qwen2.5-Coder 0.5B"}
METHOD = {"random": "Random search", "ga": "Genetic algorithm", "tpe": "TPE", "coordinate_ascent": "Coordinate ascent",
          "coevolution": "Coevolutionary GA", "nsga2": "NSGA-II", "mo_tpe": "MO-TPE"}
COLOR = {"random": "C7", "ga": "C0", "coevolution": "C1", "tpe": "C2", "coordinate_ascent": "C3", "nsga2": "C0", "mo_tpe": "C2"}

In [2]:
def check(fig):
    """Stop if any text is not 9 pt, leaves the figure, overlaps another text, enters another panel or covers a point."""
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    hidden = set()   # labels of ticks outside the axis limits exist but are not drawn
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            drawn = axis._update_ticks() if ax.axison else []
            hidden |= {l for t in axis.get_major_ticks() + axis.get_minor_ticks() if t not in drawn for l in (t.label1, t.label2)}
    texts = [t for t in fig.findobj(Text) if t.get_visible() and t.get_text().strip() and t not in hidden]
    sizes = {t.get_fontsize() for t in texts}
    assert sizes == {FONT}, f"font sizes {sizes}"
    box = {t: Text.get_window_extent(t, r) for t in texts}   # text only, without annotation arrows
    page = fig.bbox
    for t, b in box.items():
        assert b.x0 >= page.x0 - 0.5 and b.x1 <= page.x1 + 0.5 and b.y0 >= page.y0 - 0.5 and b.y1 <= page.y1 + 0.5, f"outside: {t.get_text()!r}"
    texts = list(box)
    for i, a in enumerate(texts):
        for b in texts[i + 1:]:
            ov = box[a].intersection(box[a], box[b])
            assert ov is None or ov.width < 0.5 or ov.height < 0.5, f"overlap: {a.get_text()!r} / {b.get_text()!r}"
    panels = [ax for ax in fig.axes if ax.axison]
    for t, b in box.items():
        for ax in panels:
            if t.axes is not ax and t.axes is not None and t.figure is fig:
                ov = b.intersection(b, ax.bbox)
                assert ov is None or ov.width < 0.5 or ov.height < 0.5, f"{t.get_text()!r} runs into another panel"
    for ax in fig.axes:
        pts = np.vstack([c.get_offset_transform().transform(c.get_offsets()) for c in ax.collections
                         if isinstance(c, matplotlib.collections.PathCollection)] or [np.empty((0, 2))])
        covers = [box[t] for t in box if t.axes is ax and t not in ax.get_xticklabels() + ax.get_yticklabels()]
        if ax.get_legend():
            covers.append(ax.get_legend().get_window_extent(r))
        shapes = [c.get_transform().transform_path(q) for c in ax.collections if isinstance(c, matplotlib.collections.PolyCollection) for q in c.get_paths()]
        lines = [l.get_transform().transform_path(l.get_path()) for l in ax.lines]
        for b in covers:
            assert not b.count_contains(pts), "a text or legend covers a plotted point"
            assert not any(q.intersects_bbox(b, filled=True) for q in shapes), "a text or legend covers a shaded band"
            assert not any(q.intersects_bbox(b, filled=False) for q in lines), "a text or legend covers a line"


def save(fig, name):
    check(fig)
    w, h = fig.get_size_inches()
    fig.savefig(OUT / f"{name}.pdf")
    print(f"{name}.pdf  {w / PT:.2f} pt x {h / PT:.2f} pt")
    plt.show()

## Figure 1: study design

In [3]:
fig, ax = plt.subplots(figsize=(TEXT, 2.0), layout="none")
fig.subplots_adjust(0, 0, 1, 1)
W, H = TEXT, 2.0
ax.set(xlim=(0, W), ylim=(0, H), aspect="equal")
ax.axis("off")
gap, bh = 0.30, 0.78
bw = (W - 0.04 - 3 * gap) / 4
x = [0.02 + i * (bw + gap) for i in range(4)]
top, bottom = H - 0.02 - bh, 0.02
steps = [  # (column, row, text, is data)
    (0, top, f"{key['dev_questions']} dev questions\n(synthetic,\nreferences\nrepaired)", True),
    (1, top, f"{len(grid)} configurations,\none request\nat a time", False),
    (2, top, "Judge chosen by\nblind ratings,\nRAGAS scoring", False),
    (3, top, "Lookup table:\nF1, latency and\nVRAM per answer", True),
    (3, bottom, "Search replay:\nGA, coevolution,\nTPE, CA, random;\nNSGA-II, MO-TPE", False),
    (2, bottom, "Leave-one-\nmanual-out\nvalidation", False),
    (1, bottom, f"Freeze {key['frozen_finalists']} finalists\nwith hashes of\nprotocol, code\nand data", False),
    (0, bottom, f"Score once on\n{key['test_a_questions']} unused and\n{key['test_b_questions']} independent\nquestions", True),
]
boxes, labels = [], []
for col, y, txt, data in steps:
    p = FancyBboxPatch((x[col], y), bw, bh, boxstyle="round,pad=0,rounding_size=0.05",
                       fc=(*matplotlib.colors.to_rgb("C0"), 0.15) if data else "white", ec="0.25", lw=0.8)
    ax.add_patch(p)
    boxes.append(p)
    labels.append(ax.text(x[col] + bw / 2, y + bh / 2, txt, ha="center", va="center", linespacing=1.15))
arrow = dict(arrowstyle="-|>", mutation_scale=9, color="0.25", lw=0.8, shrinkA=0, shrinkB=0)
mid_top, mid_bottom, gap_y = top + bh / 2, bottom + bh / 2, (top + bottom + bh) / 2
for i in range(3):   # dev -> grid -> judge -> table
    ax.add_patch(FancyArrowPatch((x[i] + bw, mid_top), (x[i + 1], mid_top), **arrow))
ax.add_patch(FancyArrowPatch((x[3] + bw / 2, top), (x[3] + bw / 2, bottom + bh), **arrow))     # table -> search
ax.add_patch(FancyArrowPatch((x[3], mid_bottom), (x[2] + bw, mid_bottom), **arrow))           # search -> LOMO
ax.add_patch(FancyArrowPatch((x[1], mid_bottom), (x[0] + bw, mid_bottom), **arrow))           # freeze -> test
route = MPath([(x[3] + 0.25 * bw, top), (x[3] + 0.25 * bw, gap_y), (x[1] + bw / 2, gap_y), (x[1] + bw / 2, bottom + bh)])
ax.add_patch(FancyArrowPatch(path=route, **arrow))                                           # table -> freeze

fig.canvas.draw()
r = fig.canvas.get_renderer()
for t, p in zip(labels, boxes):   # every label stays inside its box with some margin
    tb, pb = t.get_window_extent(r), p.get_window_extent(r)
    assert tb.x0 > pb.x0 + 3 and tb.x1 < pb.x1 - 3 and tb.y0 > pb.y0 + 3 and tb.y1 < pb.y1 - 3, t.get_text()
save(fig, "fig_design")

fig_design.pdf  455.24 pt x 144.54 pt


## Figure 2: where the score comes from

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(TEXT, 2.35), width_ratios=[1, 1.1])
vmin, vmax = grid.f1.mean() - 0.12, grid.f1.mean() + 0.12
for ax, row, label in zip(axes, ("chunk_size", "model"), ("Chunk size (characters)", None)):
    pv = grid.pivot_table(index=row, columns="top_k", values="f1")
    if row == "model":
        pv = pv.loc[pv.mean(axis=1).sort_values().index]
        pv.index = [MODEL[m] for m in pv.index]
    im = ax.imshow(pv.values, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(pv.shape[1]), pv.columns)
    ax.set_yticks(range(pv.shape[0]), pv.index)
    ax.set(xlabel="Top-k (retrieved chunks)", ylabel=label or "")
fig.colorbar(im, ax=axes, label="Mean F1 over the other factors", aspect=25)
save(fig, "fig_heatmaps")

fig_heatmaps.pdf  455.24 pt x 169.83 pt


## Figure 3: quality against latency

In [5]:
def pareto(f1, lat):
    """Configurations no other configuration beats on both F1 (higher) and latency (lower)."""
    f1, lat = np.asarray(f1), np.asarray(lat)
    return np.array([not np.any((f1 >= a) & (lat <= b) & ((f1 > a) | (lat < b))) for a, b in zip(f1, lat)])


fig, ax = plt.subplots(figsize=(COLUMN, 2.7))
front = grid[pareto(grid.f1, grid.latency_s)].sort_values("latency_s")
ax.scatter(grid.latency_s, grid.f1, s=4, color="0.6", linewidths=0, label=f"All {len(grid)} configurations")
ax.step(front.latency_s, front.f1, where="post", color="C0", lw=1.2)
ax.scatter(front.latency_s, front.f1, s=12, color="C0", zorder=3, label="Pareto front")
best = grid.loc[grid.f1.idxmax()]
knee = front.loc[front.f1 / best.f1 >= 0.8].iloc[0]       # fastest front point within 80 % of the best F1
fastest = front.iloc[0]
for r, txt, xy in ((best, "Qwen3-VL 30B-A3B", (0.05, 0.405)), (knee, "Qwen3-VL 2B", (0.05, 0.355)), (fastest, "Qwen2.5-Coder 0.5B", (0.047, 0.06))):
    ax.annotate(txt, (r.latency_s, r.f1), xytext=xy, ha="left", va="center",
                arrowprops=dict(arrowstyle="-", color="0.3", lw=0.6, shrinkA=1, shrinkB=2, relpos=(0.03, 1) if r is fastest else (1, 0.5)))
ax.set(xscale="log", xlabel="Mean latency per answer (s)", ylabel="Factual correctness F1 (dev)", ylim=(0.04, 0.43))
ax.set_xticks([0.05, 0.1, 0.2, 0.5], ["0.05", "0.1", "0.2", "0.5"])
ax.minorticks_off()
fig.legend(loc="outside upper center", ncols=2, frameon=False, handletextpad=0.2, columnspacing=1.0)
save(fig, "fig_pareto")

fig_pareto.pdf  216.24 pt x 195.13 pt


## Figure 4: search on the lookup table

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(TEXT, 2.4))
for ax, mode, metric, label, order in ((axes[0], "single", "regret", "Simple regret (F1)", ["random", "ga", "coevolution", "tpe", "coordinate_ascent"]),
                                       (axes[1], "multi", "hv_ratio", "Hypervolume ratio", ["random", "nsga2", "mo_tpe"])):
    g = runs[runs["mode"] == mode]
    for m in order:
        x = g[g.method == m].groupby("budget")[metric]
        med = x.median()
        ax.fill_between(med.index, x.quantile(.25), x.quantile(.75), color=COLOR[m], alpha=0.15, linewidth=0)
        ax.plot(med.index, med.values, color=COLOR[m], marker="o", ms=3, label=METHOD[m])
    ax.set(xlabel="Configurations evaluated (budget)", ylabel=label, xticks=sorted(g.budget.unique()))
    ax.legend(loc="upper right" if mode == "single" else "lower right")
axes[0].set_ylim(-0.004, 0.12)
axes[1].set_ylim(0.7, 1.0)
save(fig, "fig_optimizers")

fig_optimizers.pdf  455.24 pt x 173.45 pt


## Figure 5: held-out manuals

In [7]:
fig, ax = plt.subplots(figsize=(COLUMN, 1.95))
lo = lomo.set_index("method")
order = ["ga", "coevolution", "tpe", "coordinate_ascent"]
me = key["min_effect_f1"]
ax.axvspan(-me, me, color="0.9", linewidth=0, label=f"\u00b1{me} F1, smallest relevant effect")
ax.axvline(0, color="0.3", lw=0.6)
for i, m in enumerate(order):
    r = lo.loc[m]
    ax.errorbar(r.diff_vs_random, i, xerr=[[r.diff_vs_random - r.ci_lo], [r.ci_hi - r.diff_vs_random]], fmt="o", ms=4, color=COLOR[m], capsize=2)
ax.set_yticks(range(len(order)), [METHOD[m] for m in order])
ax.set_ylim(len(order) - 0.5, -0.5)
ax.set(xlabel="Held-out F1 minus\nrandom search", xticks=[-0.04, 0, 0.04])
fig.legend(loc="outside upper center", frameon=False)
save(fig, "fig_lomo")

fig_lomo.pdf  216.24 pt x 140.93 pt


## Figure 6: frozen finalists on untouched questions

In [8]:
fig, ax = plt.subplots(figsize=(COLUMN, 3.0))
for split, color, n in (("test_a", "C0", key["test_a_questions"]), ("test_b", "C1", key["test_b_questions"])):
    x = gen[gen.split == split]
    label = "Unused synthetic questions" if split == "test_a" else "Independently written questions"
    ax.scatter(x.dev, x.test, s=10, color=color, linewidths=0, label=f"{label} ({n})")
    b = x[x.best_dev]
    ax.scatter(b.dev, b.test, s=45, facecolors="none", edgecolors=color, linewidths=1.0)
lim = [min(gen.dev.min(), gen.test.min()) - 0.02, max(gen.dev.max(), gen.test.max()) + 0.02]
ax.plot(lim, lim, color="0.5", lw=0.8)
ax.set(xlim=lim, ylim=lim, xlabel="F1 on the dev questions", ylabel="F1 on untouched questions", aspect="equal")
fig.legend(loc="outside upper center", frameon=False, handletextpad=0.2)
save(fig, "fig_generalization")

fig_generalization.pdf  216.24 pt x 216.81 pt
